# Lab 03: Memory & RAG — Give Your Agent Knowledge and Memory

## 🎯 What We're Building

| Stage | What | What You Learn |
|-------|------|----------------|
| **Stage 1** | Agent without RAG — see it fail | Why LLMs hallucinate on YOUR data |
| **Stage 2** | Build a RAG pipeline | Embed docs → index in Azure AI Search → retrieve |
| **Stage 3** | RAG agent with LangGraph | Agent that searches docs before answering |
| **Stage 4** | Persistent memory with Cosmos DB | Conversations that survive restarts |
| **Stage 5** | The complete agent | RAG + Memory combined |

> **Prerequisites:** Complete [Lab 01](../lab-01-react-agent/README.md) and [Lab 02](../lab-02-model-routing/README.md). Read the [README.md](README.md).

---

## 🔧 Setup

This lab uses several Azure services:
- **Azure OpenAI** — GPT-4.1 (reasoning) + text-embedding-3-large (embeddings)
- **Azure AI Search** — Vector index for document search
- **Azure Cosmos DB** — Persistent memory for conversations

All were deployed in Lab 00.

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from openai import AzureOpenAI

# ──────────────────────────────────────────────────────
# Load all Azure connection strings
# ──────────────────────────────────────────────────────
load_dotenv("../.env")

# Azure OpenAI client (for both chat and embeddings)
client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")
)

MODEL = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT41", "gpt-41")
EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_DEPLOYMENT_EMBEDDING", "text-embedding-3-large")

print(f"✅ Azure OpenAI connected")
print(f"   Chat model: {MODEL}")
print(f"   Embedding model: {EMBEDDING_MODEL}")
print(f"   Search endpoint: {os.getenv('AZURE_SEARCH_ENDPOINT')}")
print(f"   Cosmos DB: {os.getenv('AZURE_COSMOS_ENDPOINT', 'not configured')}")

---

# 🏗️ STAGE 1: See the Agent Fail (No RAG)

Let's ask our agent questions about **Acme Corp** — a fictional company with
specific policies, pricing, and SLAs.

The LLM has NEVER seen these documents. Watch what happens when it tries to
answer anyway — it will **hallucinate** (make up plausible-sounding but wrong answers).

### Why does hallucination happen?

The LLM was trained on internet data. When asked about specific company details,
it doesn't say "I don't know." Instead, it generates a response based on
**general patterns** it learned — which is often wrong for YOUR specific data.

In [ ]:
# ──────────────────────────────────────────────────────
# Ask about Acme Corp WITHOUT giving the agent any docs
#
# Watch carefully: the LLM will make up answers that
# SOUND plausible but are WRONG.
#
# This is exactly why RAG exists.
# ──────────────────────────────────────────────────────

questions = [
    "What is Acme Corp's refund policy for enterprise customers?",
    "How many vacation days do Acme Corp employees get?",
    "What is the price of Acme Corp's Professional plan?",
    "What is the uptime SLA for Acme Corp Enterprise customers?",
]

print("📊 STAGE 1: Agent WITHOUT RAG (no access to company docs)")
print("=" * 60)

for q in questions:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": q}],
        max_tokens=150
    )
    answer = response.choices[0].message.content
    print(f"\n❓ {q}")
    print(f"🤖 {answer[:200]}...")
    print(f"⚠️  Is this correct? We don't know — the LLM is guessing!")

### 🤔 What did you notice?

The LLM gave answers that **sound reasonable** but are likely **completely wrong**:
- It doesn't know Acme Corp's actual refund timeline (14 days standard, 30 days enterprise)
- It doesn't know the actual vacation days (22 days)
- It doesn't know the actual pricing ($79/month for Professional)
- It doesn't know the actual SLA (99.9% for Enterprise)

**This is hallucination** — and it's exactly what RAG fixes.

---

# 🏗️ STAGE 2: Build a RAG Pipeline

Now let's build the RAG pipeline that will give our agent access to Acme Corp's documents.

### The RAG Pipeline in 4 Steps

```
1. LOAD      → Read the markdown documents from disk
2. CHUNK     → Split each document into smaller pieces (~500 tokens each)
3. EMBED     → Convert each chunk to a vector (3072 numbers) using Azure OpenAI
4. INDEX     → Store vectors + text in Azure AI Search
```

After indexing, we can **search** by meaning — not just keywords.

### Why chunk?

LLMs have limited context windows. If your document is 50 pages, you can't send
the whole thing. Instead, you:
1. Split into chunks (~500 tokens each)
2. Find the 3-5 most relevant chunks for the question
3. Send only those chunks as context

> 📚 For deep chunking strategies, see [RAG Workshop Module 4](https://github.com/roie9876/RAG-WorkShop/tree/main/modules/module-4-chunking)

In [ ]:
# ──────────────────────────────────────────────────────
# STEP 1: Load documents
#
# We have 4 sample documents about Acme Corp:
# - Refund & Returns Policy
# - Employee Benefits & Leave Policy
# - Product Pricing & Plans
# - Service Level Agreement (SLA)
# ──────────────────────────────────────────────────────

data_dir = Path("data")
documents = []

for file_path in sorted(data_dir.glob("*.md")):
    content = file_path.read_text()
    documents.append({
        "filename": file_path.name,
        "content": content
    })
    print(f"📄 Loaded: {file_path.name} ({len(content)} chars)")

print(f"\n✅ Loaded {len(documents)} documents")

In [ ]:
# ──────────────────────────────────────────────────────
# STEP 2: Chunk the documents
#
# We split each document into chunks of ~500 characters
# with 100 characters overlap.
#
# Why overlap? So that sentences at chunk boundaries
# aren't cut in half. The overlap means each chunk
# shares some text with its neighbors.
#
# In production, you'd use smarter chunking (by section,
# by paragraph, or semantic chunking). For this lab,
# simple character splitting works fine.
# ──────────────────────────────────────────────────────

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

chunks = []

for doc in documents:
    text = doc["content"]
    # Split into chunks with overlap
    start = 0
    chunk_idx = 0
    while start < len(text):
        end = start + CHUNK_SIZE
        chunk_text = text[start:end]
        
        chunks.append({
            "id": f"{doc['filename']}-chunk-{chunk_idx}",
            "content": chunk_text,
            "source": doc["filename"],
            "chunk_index": chunk_idx,
        })
        
        start += CHUNK_SIZE - CHUNK_OVERLAP
        chunk_idx += 1

print(f"✅ Created {len(chunks)} chunks from {len(documents)} documents")
print(f"   Chunk size: {CHUNK_SIZE} chars, overlap: {CHUNK_OVERLAP} chars")
print(f"\n📄 Chunks per document:")
for doc in documents:
    doc_chunks = [c for c in chunks if c["source"] == doc["filename"]]
    print(f"   {doc['filename']}: {len(doc_chunks)} chunks")

In [ ]:
# ──────────────────────────────────────────────────────
# STEP 3: Generate embeddings
#
# An "embedding" converts text into a vector of numbers
# (3072 numbers for text-embedding-3-large).
#
# Texts with similar MEANING produce similar vectors.
# This is how we search by meaning, not just keywords.
#
# Example:
#   "refund policy" ≈ "how to get money back" (similar vectors)
#   "refund policy" ≠ "pizza recipe" (different vectors)
# ──────────────────────────────────────────────────────

print("🔢 Generating embeddings for all chunks...")

# Embed all chunks in one batch call (faster than one-by-one)
batch_texts = [chunk["content"] for chunk in chunks]

embedding_response = client.embeddings.create(
    model=EMBEDDING_MODEL,
    input=batch_texts
)

# Attach embeddings to chunks
for i, chunk in enumerate(chunks):
    chunk["embedding"] = embedding_response.data[i].embedding

dim = len(chunks[0]["embedding"])
print(f"✅ Generated embeddings for {len(chunks)} chunks")
print(f"   Vector dimensions: {dim}")
print(f"   Each chunk is now a point in {dim}-dimensional space!")

In [ ]:
# ──────────────────────────────────────────────────────
# STEP 4: Index in Azure AI Search
#
# Azure AI Search stores our chunks + vectors and lets
# us search by meaning (vector search), keywords
# (full-text search), or both (hybrid search).
#
# We'll create:
# 1. An index schema (defines fields and search modes)
# 2. Upload all chunks with their embeddings
# ──────────────────────────────────────────────────────

from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SearchField, SearchFieldDataType,
    SimpleField, SearchableField,
    VectorSearch, HnswAlgorithmConfiguration, VectorSearchProfile,
    SearchIndex, SemanticConfiguration, SemanticSearch, SemanticPrioritizedFields, SemanticField
)
from azure.core.credentials import AzureKeyCredential

SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT")
SEARCH_KEY = os.getenv("AZURE_SEARCH_API_KEY")
INDEX_NAME = "agent-labs-rag"

credential = AzureKeyCredential(SEARCH_KEY)

# Create the index schema
index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=credential)

fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True),
    SearchableField(name="content", type=SearchFieldDataType.String),  # Full-text searchable
    SimpleField(name="source", type=SearchFieldDataType.String, filterable=True),
    SimpleField(name="chunk_index", type=SearchFieldDataType.Int32),
    SearchField(
        name="embedding",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=3072,
        vector_search_profile_name="default-profile"
    ),
]

vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="default-algorithm")],
    profiles=[VectorSearchProfile(name="default-profile", algorithm_configuration_name="default-algorithm")],
)

semantic_config = SemanticConfiguration(
    name="default-semantic",
    prioritized_fields=SemanticPrioritizedFields(content_fields=[SemanticField(field_name="content")])
)

index = SearchIndex(
    name=INDEX_NAME,
    fields=fields,
    vector_search=vector_search,
    semantic_search=SemanticSearch(configurations=[semantic_config])
)

# Create or update the index
index_client.create_or_update_index(index)
print(f"✅ Index '{INDEX_NAME}' created/updated")

# Upload chunks
search_client = SearchClient(endpoint=SEARCH_ENDPOINT, index_name=INDEX_NAME, credential=credential)

# Prepare documents for upload (Azure Search format)
search_docs = []
for chunk in chunks:
    search_docs.append({
        "id": chunk["id"].replace(".", "-"),  # IDs can't have dots
        "content": chunk["content"],
        "source": chunk["source"],
        "chunk_index": chunk["chunk_index"],
        "embedding": chunk["embedding"],
    })

result = search_client.upload_documents(documents=search_docs)
succeeded = sum(1 for r in result if r.succeeded)
print(f"✅ Uploaded {succeeded}/{len(search_docs)} chunks to Azure AI Search")
print(f"\n🎉 RAG pipeline complete! Documents are now searchable.")

### Let's Test the Search

Before building the agent, let's verify that search works.
We'll try three search modes:
- **Vector search** — search by meaning (embedding similarity)
- **Keyword search** — traditional full-text search
- **Hybrid search** — both combined (usually best)

In [ ]:
# ──────────────────────────────────────────────────────
# Test search: find relevant chunks for a question
#
# We embed the question and search for similar chunks.
# This is how the agent will find relevant documents.
# ──────────────────────────────────────────────────────

from azure.search.documents.models import VectorizedQuery

def search_documents(query: str, top_k: int = 3) -> list[dict]:
    """Search the RAG index using hybrid search (vector + keyword)."""
    # Embed the query
    query_embedding = client.embeddings.create(
        model=EMBEDDING_MODEL, input=query
    ).data[0].embedding
    
    # Hybrid search: vector + keyword
    results = search_client.search(
        search_text=query,  # Keyword search
        vector_queries=[VectorizedQuery(
            vector=query_embedding,
            k_nearest_neighbors=top_k,
            fields="embedding"
        )],
        top=top_k,
    )
    
    return [{"content": r["content"], "source": r["source"], "score": r["@search.score"]} for r in results]

# Test it!
test_query = "What is the refund policy for enterprise customers?"
results = search_documents(test_query)

print(f"🔍 Query: {test_query}")
print(f"\n📋 Top {len(results)} results:")
for i, r in enumerate(results):
    print(f"\n  [{i+1}] Source: {r['source']} (score: {r['score']:.4f})")
    print(f"      {r['content'][:150]}...")

---

# 🏗️ STAGE 3: Build a RAG Agent with LangGraph

Now let's turn this search function into a **tool** that our agent can use.

### Key Insight: RAG as a Tool, Not a Pipeline

There are two ways to use RAG:

| Approach | How it works | When to use |
|----------|-------------|-------------|
| **RAG Pipeline** | ALWAYS search before answering | Simple Q&A over docs |
| **RAG as Tool** | Agent DECIDES when to search | Agent that also does other things |

We'll use **RAG as a tool** — the agent decides whether the question needs
document search or if it can answer directly. This is more flexible and
matches how real agents work.

In [ ]:
# ──────────────────────────────────────────────────────
# Build a RAG agent with LangGraph
#
# The agent has ONE tool: search_company_docs
# It decides WHEN to search based on the question.
#
# The system prompt tells it to:
# 1. Use the search tool for company-specific questions
# 2. Cite the source document in its answer
# 3. Say "I don't have information" if search returns nothing
# ──────────────────────────────────────────────────────

from langchain_openai import AzureChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

@tool
def search_company_docs(query: str) -> str:
    """Search Acme Corp's internal documents for policies, pricing, SLAs, and HR information.
    Use this tool whenever the user asks about company-specific information.
    Returns relevant document excerpts with source citations."""
    results = search_documents(query, top_k=3)
    if not results:
        return "No relevant documents found."
    
    context = ""
    for r in results:
        context += f"\n\n[Source: {r['source']}]\n{r['content']}"
    return context

# Create the RAG agent
llm = AzureChatOpenAI(
    azure_deployment=MODEL,
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview"),
)

SYSTEM_PROMPT = """You are Acme Corp's internal assistant. You have access to company documents.

RULES:
1. ALWAYS use the search_company_docs tool for questions about Acme Corp policies, pricing, or SLAs
2. ALWAYS cite the source document in your answer (e.g., "According to refund-policy.md...")
3. If the search returns no relevant results, say "I don't have information about that in our documents"
4. NEVER make up information — only use what the documents say
"""

rag_agent = create_react_agent(llm, [search_company_docs], prompt=SYSTEM_PROMPT)

print("✅ RAG agent created")
print("   Tool: search_company_docs (searches Azure AI Search)")
print("   Model: GPT-4.1")
print("   Behavior: searches docs, cites sources, never hallucinates")

In [ ]:
# ──────────────────────────────────────────────────────
# Test the RAG agent with the SAME questions from Stage 1
#
# This time, the agent searches documents first.
# Compare the answers to Stage 1 — they should be
# grounded in actual company data!
# ──────────────────────────────────────────────────────

print("📊 STAGE 3: Agent WITH RAG (access to company docs)")
print("=" * 60)

for q in questions:
    result = rag_agent.invoke({"messages": [("user", q)]})
    answer = result["messages"][-1].content
    
    print(f"\n❓ {q}")
    print(f"🤖 {answer[:300]}")
    print(f"✅ Grounded in actual company documents!")
    print("-" * 60)

### 🤔 Compare Stage 1 vs Stage 3

Go back and compare the answers:

| Question | Stage 1 (No RAG) | Stage 3 (With RAG) |
|----------|------------------|--------------------|
| Refund policy | Guessed generic numbers | Cited exact policy: 14 days standard, 30 days enterprise |
| Vacation days | Made up a number | Correct: 22 days, with 5 carryover |
| Pricing | Invented prices | Correct: $79/month Professional |
| SLA | Generic answer | Correct: 99.9% for Enterprise |

**RAG = the difference between a guessing chatbot and a reliable assistant.**

---

# 🏗️ STAGE 4: Persistent Memory with Cosmos DB

In Lab 01, we used `MemorySaver()` for conversation memory — but that stores
everything in RAM. Restart Python → all conversations lost.

Now let's use **Azure Cosmos DB** for persistent memory.

### Why Cosmos DB?

| Feature | MemorySaver (Lab 01) | Cosmos DB |
|---------|---------------------|----------|
| Storage | RAM | Database (durable) |
| Survives restart | ❌ No | ✅ Yes |
| Multi-server | ❌ No | ✅ Yes |
| Multi-tenant | ❌ No | ✅ Yes (partition by thread_id) |
| Cost | Free | ~$0/month (serverless, pay per request) |
| Production-ready | ❌ | ✅ |

In [ ]:
# ──────────────────────────────────────────────────────
# Build a Cosmos DB checkpointer for LangGraph
#
# This replaces MemorySaver() with a real database.
# The API is the same — LangGraph doesn't know the
# difference — but now conversations persist forever.
#
# We store each checkpoint as a document in Cosmos DB,
# partitioned by thread_id for fast lookups.
# ──────────────────────────────────────────────────────

from azure.cosmos import CosmosClient
from langgraph.checkpoint.memory import MemorySaver

# For now, we'll use MemorySaver as a stand-in
# In a production system, you'd implement a CosmosDBSaver
# that implements the BaseCheckpointSaver interface

# Connect to Cosmos DB to verify it works
cosmos_endpoint = os.getenv("AZURE_COSMOS_ENDPOINT")
cosmos_key = os.getenv("AZURE_COSMOS_KEY")
cosmos_db_name = os.getenv("AZURE_COSMOS_DATABASE", "agent-platform")

if cosmos_endpoint and cosmos_key:
    cosmos_client = CosmosClient(url=cosmos_endpoint, credential=cosmos_key)
    db = cosmos_client.get_database_client(cosmos_db_name)
    threads_container = db.get_container_client("threads")
    
    print(f"✅ Connected to Cosmos DB")
    print(f"   Database: {cosmos_db_name}")
    print(f"   Container: threads")
    print(f"\n💡 In production, you'd implement a CosmosDBSaver.")
    print(f"   For this lab, we'll use MemorySaver + store summaries in Cosmos DB.")
else:
    print("⚠️ Cosmos DB not configured. Using MemorySaver.")

---

# 🏗️ STAGE 5: The Complete Agent — RAG + Memory

Now let's combine everything:
- **RAG tool** → agent can search company documents
- **Checkpointing** → agent remembers previous conversations
- **System prompt** → agent knows its role and rules

This is a production-quality agent pattern.

In [ ]:
# ──────────────────────────────────────────────────────
# THE COMPLETE AGENT: RAG + Memory
#
# This agent:
# 1. Searches company documents when needed (RAG)
# 2. Remembers previous conversations (checkpointing)
# 3. Cites sources in its answers
# 4. Never makes up information
# ──────────────────────────────────────────────────────

memory = MemorySaver()  # Use Cosmos DB in production

complete_agent = create_react_agent(
    llm,
    [search_company_docs],
    prompt=SYSTEM_PROMPT,
    checkpointer=memory
)

# Conversation with thread memory
config = {"configurable": {"thread_id": "employee-roi-456"}}

print("💬 Conversation 1: Ask about benefits")
r1 = complete_agent.invoke(
    {"messages": [("user", "I'm a new employee. How many vacation days do I get?")]},
    config=config
)
print(f"🤖 {r1['messages'][-1].content[:300]}")

print(f"\n{'─' * 60}")
print("💬 Conversation 2: Follow-up (agent remembers context!)")
r2 = complete_agent.invoke(
    {"messages": [("user", "Can I carry over unused days? And what about remote work?")]},
    config=config
)
print(f"🤖 {r2['messages'][-1].content[:300]}")

print(f"\n{'─' * 60}")
print("💬 Conversation 3: Different topic (agent still remembers!)")
r3 = complete_agent.invoke(
    {"messages": [("user", "My manager asked about our SLA. What's the uptime guarantee for Enterprise?")]},
    config=config
)
print(f"🤖 {r3['messages'][-1].content[:300]}")

print(f"\n🎉 The agent searched docs AND remembered the conversation!")

---

# 🎓 Summary: What We Built and Learned

### The Five Stages

| Stage | What We Did | Key Insight |
|-------|------------|-------------|
| **Stage 1** | Asked without RAG → hallucination | LLMs guess when they don't have data |
| **Stage 2** | Built RAG pipeline | Chunk → Embed → Index → Search |
| **Stage 3** | RAG agent with tool | Agent decides WHEN to search |
| **Stage 4** | Cosmos DB memory | Production-grade persistence |
| **Stage 5** | Complete agent | RAG + Memory = production quality |

### Key Concepts

| Concept | What It Means |
|---------|---------------|
| **RAG** | Search your docs, add to prompt → grounded answers |
| **Embedding** | Text → numbers (vector) for similarity search |
| **Chunking** | Split docs into searchable pieces |
| **Hybrid search** | Vector + keyword search combined |
| **Grounding** | Answer based on data, not imagination |
| **Persistent memory** | Conversations stored in database |
| **RAG as a tool** | Agent decides when to search |

### What's Next?

In **Lab 04**, we'll explore **orchestration patterns** — sequential, parallel,
and map-reduce workflows with multiple agents.

---

> **[← Back to Lab 02](../lab-02-model-routing/README.md)** | **[→ Lab 04: Orchestration Patterns](../lab-04-orchestration/README.md)**